In [ ]:
!pip -q install scikit-learn pandas numpy scipy statsmodels matplotlib
from google.colab import drive; drive.mount('/content/drive')
BASE='/content/drive/MyDrive/deprem'
TRAIN_CSV=BASE+'/annotation/final_300.csv'
CORPUS_CSV=BASE+'/corpus/corpus_final.csv'
SCORED_CSV=BASE+'/outputs/analytic_with_scores.csv'
OUT_DIR=BASE+'/outputs'
import os; os.makedirs(OUT_DIR+'/figures',exist_ok=True)
for p in [TRAIN_CSV,CORPUS_CSV,SCORED_CSV]: print(('OK   ' if os.path.exists(p) else 'EKSIK '),p)

In [ ]:
import numpy as np, pandas as pd, re
from scipy.sparse import hstack
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVR

FRAMES = ["technical","political","development","sustainability"]
RANDOM_STATE = 42
TURKISH_STOPWORDS = ["acaba","ama","ancak","artık","asla","aslında","az","bana","bazen","bazı","belki","ben","benden","beni","benim","beri","beş","bile","bir","biraz","birçok","biri","birkaç","birşey","biz","bizden","bizi","bizim","böyle","böylece","bu","buna","bunda","bundan","bunlar","bunları","bunların","bunu","bunun","burada","çok","çünkü","da","daha","dahi","de","defa","değil","diğer","diye","doksan","dokuz","dolayı","dolayısıyla","dört","elli","en","fakat","falan","filan","gene","gibi","hala","hangi","hatta","hem","henüz","hep","hepsi","her","herhangi","herkes","hiç","hiçbir","için","iki","ile","ilgili","ise","işte","itibaren","itibariyle","kadar","karşın","kendi","kendilerine","kendini","kendisi","kendisine","kendisini","kez","ki","kim","kimden","kime","kimi","kimse","madem","mı","mi","mu","mü","nasıl","ne","neden","nedenle","nerde","nerede","nereye","niçin","niye","o","olan","olarak","oldu","olduğu","olduğunu","olduklarını","olmadı","olmadığı","olmak","olması","olmayan","olmaz","olsa","olsun","olup","olur","olursa","oluyor","on","ona","ondan","onlar","onlardan","onları","onların","onu","onun","otuz","oysa","öyle","pek","rağmen","sana","sanki","sekiz","seksen","sen","senden","seni","senin","siz","sizden","sizi","sizin","sonra","şayet","şey","şeyden","şeyi","şeyler","şimdi","şu","şuna","şunda","şundan","şunları","şunu","tüm","üç","üzere","var","vardı","ve","veya","ya","yani","yedi","yerine","yetmiş","yine","yirmi","yoksa","yüz","zaten"]

def make_vectorizers():
    wv = TfidfVectorizer(analyzer="word", ngram_range=(1,2), max_features=20000, min_df=3,
                         stop_words=TURKISH_STOPWORDS, lowercase=True, sublinear_tf=True)
    cv = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=20000, min_df=3,
                         lowercase=True, sublinear_tf=True)
    return wv, cv

train  = pd.read_csv(TRAIN_CSV, encoding="utf-8-sig")
corpus = pd.read_csv(CORPUS_CSV, encoding="utf-8-sig"); corpus["text"]=corpus["text"].fillna("").astype(str)
scored = pd.read_csv(SCORED_CSV, encoding="utf-8-sig")
print("yüklendi:", len(train), "train,", len(corpus), "corpus,", len(scored), "scored")

In [ ]:
# === 1) HUMAN-ONLY ROBUSTNESS ===
rows=[]
for frame in FRAMES:
    sub = train[(train[frame+"_source"]=="human_agreement") & train["text"].notna() & train[frame].notna()].copy()
    if len(sub)<20:
        print(f"[{frame}] {len(sub)} human doc — atlandı"); continue
    wv,cv = make_vectorizers()
    Xtr = hstack([wv.fit_transform(sub["text"]), cv.fit_transform(sub["text"])]).tocsr()
    m = LinearSVR(random_state=RANDOM_STATE, max_iter=10000).fit(Xtr, sub[frame].astype(float))
    Xall = hstack([wv.transform(corpus["text"]), cv.transform(corpus["text"])]).tocsr()
    comp = corpus[["doc_id"]].copy(); comp["human_only"]=np.clip(m.predict(Xall),0,3)
    comp = comp.merge(scored[["doc_id",f"tfidf_score_{frame}"]], on="doc_id", how="inner")
    r,_=pearsonr(comp["human_only"], comp[f"tfidf_score_{frame}"])
    rho,_=spearmanr(comp["human_only"], comp[f"tfidf_score_{frame}"])
    shift=comp["human_only"].mean()-comp[f"tfidf_score_{frame}"].mean()
    rows.append({"frame":frame,"n_human_train":len(sub),"pearson_r":round(r,4),"spearman_rho":round(rho,4),"mean_shift":round(shift,4)})
    print(f"[{frame}] n={len(sub)} r={r:.3f} rho={rho:.3f} shift={shift:+.3f}")
hum = pd.DataFrame(rows); hum.to_csv(OUT_DIR+"/robustness_human_only.csv", index=False)
print("\nyazıldı: robustness_human_only.csv"); display(hum)

In [ ]:
# === 2) NAME-REDACTION ROBUSTNESS (Political) ===
REDACT = ["akp","ak parti","chp","mhp","iyi parti","erdoğan","erdogan","recep tayyip","kılıçdaroğlu","kilicdaroglu","imamoğlu","imamoglu","yavaş","yavas","özel","bahçeli","bahceli","soyer","böcek","bocek","cumhurbaşkanı","cumhurbaskani","cumhurbaşkanlığı","büyükşehir belediye başkanı","belediye başkanı"]
pats=[re.compile(re.escape(t), re.IGNORECASE) for t in REDACT]
def redact(s):
    for p in pats: s=p.sub(" ", s)
    return s
frame="political"
tr=train[train[frame].notna()].copy(); tr["red"]=tr["text"].fillna("").astype(str).map(redact)
corpus["red"]=corpus["text"].map(redact)
wv,cv=make_vectorizers()
Xtr=hstack([wv.fit_transform(tr["red"]), cv.fit_transform(tr["red"])]).tocsr()
m=LinearSVR(random_state=RANDOM_STATE, max_iter=10000).fit(Xtr, tr[frame].astype(float))
Xall=hstack([wv.transform(corpus["red"]), cv.transform(corpus["red"])]).tocsr()
corpus[frame+"_red"]=np.clip(m.predict(Xall),0,3)
comp=corpus[["doc_id","date",frame+"_red"]].merge(scored[["doc_id",f"tfidf_score_{frame}",f"distilbert_score_{frame}_0to3"]],on="doc_id",how="inner")
r_tf,_=pearsonr(comp[frame+"_red"],comp[f"tfidf_score_{frame}"])
r_db,_=pearsonr(comp[frame+"_red"],comp[f"distilbert_score_{frame}_0to3"])
comp["date"]=pd.to_datetime(comp["date"],errors="coerce"); comp=comp.dropna(subset=["date"])
comp["month"]=comp["date"].dt.to_period("M").dt.to_timestamp()
mon=comp.groupby("month").agg(redacted_mean=(frame+"_red","mean"),original_mean=(f"tfidf_score_{frame}","mean"),n=("doc_id","count")).reset_index()
mcorr=mon["redacted_mean"].corr(mon["original_mean"])
print(f"redacted vs TF-IDF r={r_tf:.3f} | vs DistilBERT r={r_db:.3f} | aylık pattern r={mcorr:.3f}")
mon.to_csv(OUT_DIR+"/robustness_name_redaction.csv",index=False)
summ=pd.DataFrame([{"frame":frame,"r_vs_tfidf":round(r_tf,4),"r_vs_distilbert":round(r_db,4),"monthly_pattern_corr":round(mcorr,4),"n_redact_terms":len(REDACT)}])
summ.to_csv(OUT_DIR+"/robustness_name_redaction_summary.csv",index=False)
print("yazıldı: robustness_name_redaction(_summary).csv"); display(summ)

In [ ]:
# === 3) DiD PARALLEL-TRENDS + PLACEBO ===
import statsmodels.formula.api as smf
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt; import matplotlib.dates as mdates
TREAT_DATE="2024-03-31"; PLACEBO_DATE="2023-10-01"; TREAT_TYPES=["belediye"]
SCORE="distilbert_score_{}_0to3"
d=scored.copy(); d["date"]=pd.to_datetime(d["date"],errors="coerce"); d=d.dropna(subset=["date"])
d["month"]=d["date"].dt.to_period("M").dt.to_timestamp(); d["source_type"]=d["source_type"].str.strip().str.lower()
def did_at(df,tdate):
    out=[]
    for frame in FRAMES:
        lo=pd.Timestamp(tdate)-pd.DateOffset(months=6); hi=pd.Timestamp(tdate)+pd.DateOffset(months=6)
        s=df[(df["date"]>=lo)&(df["date"]<hi)].copy()
        s["treat"]=s["source_type"].isin(TREAT_TYPES).astype(int); s["post"]=(s["date"]>=pd.Timestamp(tdate)).astype(int)
        s["y"]=s[SCORE.format(frame)]
        if s["treat"].nunique()<2 or s["post"].nunique()<2: continue
        mm=smf.ols("y ~ treat * post", data=s).fit()
        out.append({"frame":frame,"DiD_coef":round(mm.params.get("treat:post",np.nan),4),"se":round(mm.bse.get("treat:post",np.nan),4),"p":round(mm.pvalues.get("treat:post",np.nan),4),"n":int(len(s))})
    return out
real=[{**r,"event":f"REAL {TREAT_DATE}"} for r in did_at(d,TREAT_DATE)]
plac=[{**r,"event":f"PLACEBO {PLACEBO_DATE}"} for r in did_at(d,PLACEBO_DATE)]
did=pd.DataFrame(real+plac); did.to_csv(OUT_DIR+"/did_placebo_results.csv",index=False)
d["grp"]=np.where(d["source_type"].isin(TREAT_TYPES),"Local gov (treat)","Other (control)")
fig,axes=plt.subplots(2,2,figsize=(13,8),sharex=True)
for ax,frame in zip(axes.ravel(),FRAMES):
    g=d.groupby(["month","grp"])[SCORE.format(frame)].mean().reset_index()
    for grp,sub in g.groupby("grp"): ax.plot(sub["month"],sub[SCORE.format(frame)],marker="o",ms=3,label=grp)
    ax.axvline(pd.Timestamp(TREAT_DATE),color="red",ls="--",lw=1); ax.axvline(pd.Timestamp(PLACEBO_DATE),color="gray",ls=":",lw=1)
    ax.set_title(frame.capitalize()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m"))
axes[0,0].legend(fontsize=8); fig.suptitle("Treat vs control aylık ortalama (kırmızı=yerel seçim, nokta=placebo)"); fig.tight_layout()
fig.savefig(OUT_DIR+"/figures/V_did_parallel_trends.png",dpi=150,bbox_inches="tight")
print("yazıldı: did_placebo_results.csv + figures/V_did_parallel_trends.png"); display(did)

In [ ]:
# === 4) BERTurk-base vs DistilBERTurk
!pip -q install "transformers>=4.40" "datasets>=2.18" "accelerate>=0.27" scikit-learn pandas numpy

import os, shutil, numpy as np, pandas as pd, torch, torch.nn as nn
from datasets import Dataset
from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from sklearn.model_selection import KFold
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)

# ---- AYARLAR (gerekirse yolu degistir) ----------------------------------
TRAIN_CSV = '/content/drive/MyDrive/deprem/annotation/final_300.csv'
OUT_DIR   = '/content/drive/MyDrive/deprem/outputs'
FRAMES    = ["technical","political","development","sustainability"]
TEXT_COL  = "text"
RANDOM_STATE, MAX_LEN, LABEL_MAX, BATCH_SIZE = 42, 512, 3.0, 8
EPOCHS, LR, N_FOLDS = 10, 1e-5, 5
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cihaz:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (yavas!)")
assert os.path.exists(TRAIN_CSV), f"final_300.csv bulunamadi: {TRAIN_CSV} — Drive'da bu yolda oldugundan emin ol."

# ---- veri ----------------------------------------------------------------
df = pd.read_csv(TRAIN_CSV, encoding="utf-8-sig")
df = df.dropna(subset=[TEXT_COL]+FRAMES).reset_index(drop=True)
for f in FRAMES: df[f] = df[f].astype(int)
print(f"egitim: {len(df)} doc")

# ---- per-sample, per-frame ters-frekans agirliklari ----------------------
def compute_frame_weights(d, thr=5.0):
    n = len(d); w = np.ones((n, len(FRAMES)), dtype=np.float32)
    for j,f in enumerate(FRAMES):
        y = d[f].values.astype(int); cls,cnt = np.unique(y, return_counts=True)
        if len(cls)<=1 or cnt.max()/max(cnt.min(),1) < thr: continue
        k=len(cls); c2c=dict(zip(cls,cnt))
        for i,yi in enumerate(y): w[i,j]=n/(k*c2c[yi])
    return w

def to_ds(d, tok, weights):
    labels=(d[FRAMES].values.astype(np.float32))/LABEL_MAX
    ds=Dataset.from_dict({TEXT_COL:d[TEXT_COL].tolist()})
    ds=ds.map(lambda b: tok(b[TEXT_COL], truncation=True, max_length=MAX_LEN, padding=False),
              batched=True, remove_columns=[TEXT_COL])
    ds=ds.add_column("labels", labels.tolist())
    ds=ds.add_column("sample_weight", weights.tolist())
    return ds

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels=inputs.pop("labels"); sw=inputs.pop("sample_weight")
        out=model(**inputs); logits=out.logits
        loss=(((logits.float()-labels.float())**2)*sw.float()).mean()
        return (loss,out) if return_outputs else loss

def make_collate(tokenizer):
    # Dinamik padding: farkli uzunluktaki diziler (orn. 474 vs 512) tek tensore toplanir.
    def collate(features):
        sw=[f.pop("sample_weight") for f in features]
        lb=[f.pop("labels") for f in features]
        batch=tokenizer.pad(features, padding=True, return_tensors="pt")
        batch["sample_weight"]=torch.tensor(sw,dtype=torch.float32)
        batch["labels"]=torch.tensor(lb,dtype=torch.float32)
        return batch
    return collate

def metrics_fn(ep):
    preds,labels=ep; preds=np.asarray(preds,np.float32); labels=np.asarray(labels,np.float32)
    m={}
    for i,f in enumerate(FRAMES):
        m[f"mae_{f}"]=float(mean_absolute_error(labels[:,i],preds[:,i]))
        ti=np.clip(np.rint(labels[:,i]*LABEL_MAX),0,3).astype(int)
        pi=np.clip(np.rint(preds[:,i]*LABEL_MAX),0,3).astype(int)
        m[f"qwk_{f}"]=float(cohen_kappa_score(ti,pi,weights="quadratic",labels=[0,1,2,3]))
    return m

def run_cv(hf_model, name):
    tok=AutoTokenizer.from_pretrained(hf_model)
    kf=KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    pf_mae={f:[] for f in FRAMES}; pf_qwk={f:[] for f in FRAMES}
    for fold,(tr,te) in enumerate(kf.split(df),1):
        print(f"  [{name}] fold {fold}/{N_FOLDS}")
        tr_df=df.iloc[tr].reset_index(drop=True); te_df=df.iloc[te].reset_index(drop=True)
        trw=compute_frame_weights(tr_df); tew=np.ones((len(te_df),len(FRAMES)),np.float32)
        model=AutoModelForSequenceClassification.from_pretrained(hf_model, num_labels=len(FRAMES), problem_type="regression")
        args=TrainingArguments(output_dir=f"/content/_cv_{name}_{fold}", num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE*2,
            learning_rate=LR, weight_decay=0.01, warmup_ratio=0.1, eval_strategy="epoch",
            save_strategy="no", logging_steps=50, seed=RANDOM_STATE, report_to="none",
            fp16=torch.cuda.is_available(), disable_tqdm=True, remove_unused_columns=False)
        tr_ds=to_ds(tr_df,tok,trw); te_ds=to_ds(te_df,tok,tew)
        trainer=WeightedTrainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=te_ds,
                                data_collator=make_collate(tok), compute_metrics=metrics_fn)
        trainer.train(); ev=trainer.evaluate()
        for f in FRAMES: pf_mae[f].append(ev[f"eval_mae_{f}"]); pf_qwk[f].append(ev[f"eval_qwk_{f}"])
        shutil.rmtree(f"/content/_cv_{name}_{fold}", ignore_errors=True)
        del trainer, model; torch.cuda.empty_cache()
    rows=[]
    for f in FRAMES:
        rows.append({"experiment":name,"frame":f,
                     "mae_mean":round(np.mean(pf_mae[f]),4),"mae_std":round(np.std(pf_mae[f]),4),
                     "qwk_mean":round(np.mean(pf_qwk[f]),4),"qwk_std":round(np.std(pf_qwk[f]),4)})
    macro_qwk=np.mean([np.mean(pf_qwk[f]) for f in FRAMES]); macro_mae=np.mean([np.mean(pf_mae[f]) for f in FRAMES])
    rows.append({"experiment":name,"frame":"MEAN","mae_mean":round(macro_mae,4),"mae_std":0,
                 "qwk_mean":round(macro_qwk,4),"qwk_std":0})
    print(f"  [{name}] MACRO QWK={macro_qwk:.3f} MAE={macro_mae:.3f}")
    return rows

all_rows=[]
for name,hf in [("berturk_base_weighted","dbmdz/bert-base-turkish-cased"),
                ("distilberturk_weighted","dbmdz/distilbert-base-turkish-cased")]:
    print(f"\n########## {name} ({hf}) ##########")
    all_rows += run_cv(hf, name)

res=pd.DataFrame(all_rows)
res.to_csv(OUT_DIR+"/w5_berturk_comparison.csv", index=False)
print("\nyazildi:", OUT_DIR+"/w5_berturk_comparison.csv")
try:
    display(res[res.frame=="MEAN"])
    display(res)
except: print(res.to_string(index=False))
